# Usage of PySpark SQL

In [4]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                     .appName("Analyzing an unknown article.")
                     .getOrCreate())


The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.

/opt/anaconda3/envs/cs326/lib/python3.10/site-packages/pyspark/bin/spark-class: line 97: CMD: bad array subscript
head: illegal line count -- -1


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [3]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 18.2 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008643 sha256=df85daaaec0b887f8b6441af9906184f50637527e8b7e3965c43ff2766dfb6c4
  Stored in directory: /Users/brookshu/Library/Caches/pip/wheels/16/77/d3/d15aaaab1df8384ad9bd94caba26a1a5aa439d8afd187a5ab9
Successfully built pyspark


In [ ]:
spark

In [ ]:
sc = spark.sparkContext

In [ ]:
## documentation
spark.read??

In [ ]:
file_path = r' '

In [ ]:
article = spark.read.text(file_path)

In [ ]:
article

In [ ]:
article.printSchema()

In [ ]:
article.select(article.value)

In [ ]:
article.show(5, truncate=False)

In [ ]:
from pyspark.sql.functions import col

In [ ]:

article.select(article.value)
article.select(article['value'])
article.select(col('value'))
article.select('value')

In [ ]:
from pyspark.sql.functions import col, split

lines = article.select(
    split(col('value'), " ").alias('line')
)

In [ ]:
lines.printSchema()

In [ ]:
lines.show(5, truncate=False)

In [ ]:
lines

In [ ]:
from pyspark.sql.functions import explode

words = lines.select(explode(col("line")).alias('word'))

In [ ]:
words.printSchema()

In [ ]:
from pyspark.sql.functions import lower

words_lower = words.select(lower(col("word")).alias('word_lower'))

In [ ]:
words_lower.show(10)

In [ ]:
from pyspark.sql.functions import regexp_extract

words_clean = words_lower.select(
    regexp_extract(col("word_lower"), r"(\W+)?([a-z]+)", 2).alias("word_clean")
)

In [ ]:
words_clean.show(10)

In [ ]:
words_nonull = words_clean.where(col("word_clean") != "")

words_nonull.show(100)

In [ ]:
groups = words_nonull.groupBy(col("word_clean"))

In [ ]:
groups

In [ ]:
counts = groups.count()

In [ ]:
counts.orderBy('count', ascending=False).show(10)

In [ ]:
import pyspark.sql.functions as F

counts = (
    spark.read.text(file_path)
     .select(F.split(F.col('value'), ' ').alias('line'))
     .select(F.explode(F.col('line')).alias('word'))
     .select(F.lower(F.col('word')).alias('word'))
     .select(F.regexp_extract(F.col('word'), r"(\W+)?([a-z]+)", 2).alias('word'))
     .where(F.col('word') != "")
     .groupby('word')
     .count()
)

In [ ]:
counts.show(10)